# GENOME FORENSICS AI
### DNA sequence fingerprinting, mutation comparison, similarity networks & anomaly screening

**Goal:** Explore sequence-level similarities and unusual patterns in DNA records using reproducible bioinformatics methods.

**Important:** An anomaly is only a computational flag. It is not proof of contamination, engineering, malicious activity, transmission, or common ancestry. Similarity alone cannot establish evolutionary direction or causation.

**Demo data:** The default sequences are short illustrative examples, not biological evidence. Replace them with curated public FASTA records for research.

## 1. Install dependencies
Run once in a notebook environment.

In [ ]:
%pip -q install biopython pandas numpy matplotlib seaborn scikit-learn networkx

In [ ]:
import re, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import networkx as nx
from Bio import SeqIO, pairwise2
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
sns.set_theme()
RANDOM_STATE = 42
DNA = set("ACGT")
print("Environment ready")

## 2. Load FASTA sequences
Set `FASTA_PATH` to your FASTA file. If `None`, a demonstration dataset is used.

In [ ]:
FASTA_PATH = None  # Example: "data/sequences.fasta"
demo = {
 "demo_01":"ATGCGTACGTTAGCGTACGATCGATCGTACGTTAGC",
 "demo_02":"ATGCGTACGTTAGCGTACGATCGATCGTACGTTAGT",
 "demo_03":"ATGCGTACGTTAGCATACGATCGATCGTACGTTAGC",
 "demo_04":"TTGCGTACGTTAGCATACGATCGATCGTACGTTAGC",
 "demo_05":"GGGAAACCCGGGTTTAAACCCGGGTTTAAACCCGGG",
 "demo_06":"GGGAAACCCGGGTTTAAACCCGGGTTTAAACCCGGA",
 "demo_07":"CCCCAAAATTTTGGGGCCCCAAAATTTTGGGGCCCC",
 "demo_08":"CCCCAAAATTTTGGGGCCCCAAAATTTTGGGGCCCA",
}
if FASTA_PATH and Path(FASTA_PATH).exists():
    records = list(SeqIO.parse(FASTA_PATH, "fasta"))
    sequences = {r.id: str(r.seq).upper().replace("U","T") for r in records}
    DATA_SOURCE = FASTA_PATH
else:
    sequences = demo.copy()
    DATA_SOURCE = "Illustrative built-in demo sequences"
print(DATA_SOURCE, "|", len(sequences), "sequences")

## 3. Validate and summarize sequences

In [ ]:
def clean_and_check(raw):
    seq = re.sub(r"\\s+", "", str(raw).upper()).replace("U","T")
    invalid = sorted(set(seq) - DNA)
    return seq, invalid

clean = {}
validation = []
for sid, raw in sequences.items():
    seq, invalid = clean_and_check(raw)
    clean[sid] = seq
    validation.append({"sequence_id":sid, "length_bp":len(seq),
                       "invalid_symbols":"".join(invalid),
                       "valid_acgt_only":not invalid and bool(seq)})
validation_df = pd.DataFrame(validation)

def summarize(sid, seq):
    c = Counter(seq); n = sum(c[b] for b in DNA)
    return {"sequence_id":sid, "length_bp":len(seq),
            "A":c["A"],"C":c["C"],"G":c["G"],"T":c["T"],
            "GC_percent":100*(c["G"]+c["C"])/n if n else np.nan,
            "ambiguous_or_other":len(seq)-n}
summary_df = pd.DataFrame([summarize(k,v) for k,v in clean.items()])
display(validation_df)
display(summary_df)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(11,4))
sns.histplot(summary_df.length_bp, ax=ax[0]); ax[0].set_title("Sequence lengths")
sns.histplot(summary_df.GC_percent.dropna(), ax=ax[1]); ax[1].set_title("GC content (%)")
plt.tight_layout()

## 4. K-mer sequence fingerprints
A k-mer is a substring of length *k*. Normalized frequencies create a compact compositional profile; they are not unique identifiers.

In [ ]:
def kmer_counts(seq, k=3):
    return Counter(seq[i:i+k] for i in range(len(seq)-k+1)
                   if set(seq[i:i+k]) <= DNA)
K = 3
vocab = sorted(set().union(*(set(kmer_counts(s,K)) for s in clean.values())))
def kmer_vector(seq):
    c=kmer_counts(seq,K); total=sum(c.values())
    return {x:c.get(x,0)/total if total else 0 for x in vocab}
fingerprints = pd.DataFrame({sid:kmer_vector(seq) for sid,seq in clean.items()}).T.fillna(0)
fingerprints.index.name="sequence_id"
display(fingerprints.iloc[:,:min(12,len(vocab))])

In [ ]:
plt.figure(figsize=(max(9,len(vocab)*0.22),5))
sns.heatmap(fingerprints,cmap="viridis")
plt.title(f"Normalized {K}-mer fingerprints"); plt.xlabel("k-mer"); plt.tight_layout()

## 5. Pairwise global alignment and identity matrix
Identity is matches divided by total alignment columns, including gaps. This is a simple exploratory measure, not a phylogenetic distance model.

In [ ]:
ids=list(clean)
identity=pd.DataFrame(np.eye(len(ids)),index=ids,columns=ids)
scores=pd.DataFrame(np.nan,index=ids,columns=ids)
def alignment_identity(a,b):
    aln=pairwise2.align.globalms(a,b,2,-1,-5,-0.5,one_alignment_only=True)[0]
    aa,bb=aln.seqA,aln.seqB
    return sum(x==y for x,y in zip(aa,bb))/len(aa), aln.score
for i,a in enumerate(ids):
    for b in ids[i+1:]:
        val,score=alignment_identity(clean[a],clean[b])
        identity.loc[a,b]=identity.loc[b,a]=val
        scores.loc[a,b]=scores.loc[b,a]=score
display(identity)

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(identity,annot=True,fmt=".2f",vmin=0,vmax=1,cmap="mako")
plt.title("Pairwise global-alignment identity"); plt.tight_layout()

## 6. Sequence similarity network
Nodes represent sequences; edges connect pairs above the selected identity threshold. The graph shows similarity groups, not proof of descent or transmission.

In [ ]:
IDENTITY_THRESHOLD=0.85
G=nx.Graph()
G.add_nodes_from(ids)
for i,a in enumerate(ids):
    for b in ids[i+1:]:
        if pd.notna(identity.loc[a,b]) and identity.loc[a,b]>=IDENTITY_THRESHOLD:
            G.add_edge(a,b,weight=float(identity.loc[a,b]))
print("Nodes:",G.number_of_nodes(),"Edges:",G.number_of_edges())
print("Connected components:",[sorted(x) for x in nx.connected_components(G)])
plt.figure(figsize=(9,6))
pos=nx.spring_layout(G,seed=RANDOM_STATE,weight="weight")
nx.draw_networkx(G,pos,node_size=850,font_size=8)
plt.title(f"Similarity network (identity ≥ {IDENTITY_THRESHOLD})"); plt.axis("off")

In [ ]:
bc=nx.betweenness_centrality(G)
network_metrics=pd.DataFrame({
 "sequence_id":list(G.nodes),
 "degree":[G.degree(n) for n in G.nodes],
 "betweenness_centrality":[bc[n] for n in G.nodes]
})
display(network_metrics)

## 7. Exploratory anomaly screening
Isolation Forest flags records that look unusual relative to this particular dataset. It does not explain the cause of an outlier.

In [ ]:
features=summary_df.set_index("sequence_id")[["length_bp","GC_percent"]].join(fingerprints)
features=features.replace([np.inf,-np.inf],np.nan).fillna(0)
if len(features)>=5:
    X=StandardScaler().fit_transform(features)
    detector=IsolationForest(n_estimators=300,contamination=min(.2,max(.05,1/len(features))),
                             random_state=RANDOM_STATE)
    labels=detector.fit_predict(X)
    anomaly_df=pd.DataFrame({"sequence_id":features.index,
        "screening_result":np.where(labels==-1,"Review","Not flagged"),
        "anomaly_score":detector.decision_function(X)}).sort_values("anomaly_score")
else:
    anomaly_df=pd.DataFrame(columns=["sequence_id","screening_result","anomaly_score"])
    print("Use at least five sequences for this exploratory screen.")
display(anomaly_df)

In [ ]:
if len(features)>=5:
    coords=PCA(n_components=2,random_state=RANDOM_STATE).fit_transform(X)
    plot=pd.DataFrame(coords,index=features.index,columns=["PC1","PC2"]).join(
        anomaly_df.set_index("sequence_id")[["screening_result"]])
    plt.figure(figsize=(8,6))
    sns.scatterplot(data=plot,x="PC1",y="PC2",hue="screening_result",s=120)
    for sid,row in plot.iterrows(): plt.text(row.PC1,row.PC2,sid,fontsize=8)
    plt.title("PCA of sequence features"); plt.tight_layout()

## 8. Mutation comparison against a reference
The following reports mismatches and gaps in a global alignment. These are sequence differences, not clinical variants or evidence of functional impact.

In [ ]:
REFERENCE_ID=ids[0]
def mutation_report(ref, query):
    aln=pairwise2.align.globalms(ref,query,2,-1,-5,-0.5,one_alignment_only=True)[0]
    a,b=aln.seqA,aln.seqB
    ref_pos=0; rows=[]
    for col,(x,y) in enumerate(zip(a,b),start=1):
        if x!="-": ref_pos+=1
        if x!=y:
            event=("insertion_in_query" if x=="-" else
                   "deletion_in_query" if y=="-" else "substitution")
            rows.append({"alignment_column":col,
                         "reference_position":ref_pos if x!="-" else None,
                         "reference_base":x,"query_base":y,"event":event})
    return pd.DataFrame(rows)
mutation_tables={}
for sid,seq in clean.items():
    if sid!=REFERENCE_ID:
        mutation_tables[sid]=mutation_report(clean[REFERENCE_ID],seq)
for sid,table in mutation_tables.items():
    print(f"Reference {REFERENCE_ID} vs {sid}")
    display(table)

## 9. Evidence summary

In [ ]:
def make_summary(sid):
    row=summary_df.set_index("sequence_id").loc[sid]
    result=anomaly_df.set_index("sequence_id")["screening_result"].get(sid,"Not evaluated")
    return {"sequence_id":sid,"length_bp":int(row.length_bp),
            "GC_percent":round(float(row.GC_percent),3) if pd.notna(row.GC_percent) else None,
            "identity_to_reference":round(float(identity.loc[sid,REFERENCE_ID]),4),
            "anomaly_screen":result,
            "interpretation":"Exploratory observation; requires biological context and independent validation."}
interpretation_df=pd.DataFrame([make_summary(sid) for sid in ids])
display(interpretation_df)

## 10. Export results

In [ ]:
OUT=Path("genome_forensics_outputs")
OUT.mkdir(exist_ok=True)
exports={
 "sequence_validation.csv":validation_df,
 "sequence_summary.csv":summary_df,
 "kmer_fingerprints.csv":fingerprints,
 "pairwise_identity_matrix.csv":identity,
 "alignment_scores.csv":scores,
 "network_metrics.csv":network_metrics,
 "anomaly_screening.csv":anomaly_df,
 "interpretation_report.csv":interpretation_df
}
for name,df in exports.items(): df.to_csv(OUT/name)
for sid,df in mutation_tables.items():
    safe=re.sub(r"[^A-Za-z0-9_.-]","_",sid)
    df.to_csv(OUT/f"mutations_vs_{safe}.csv",index=False)
print("Saved",len(list(OUT.iterdir())),"files to",OUT.resolve())

## 11. Research-grade improvements and limitations

For a credible research analysis:
1. Replace demo sequences with curated public FASTA records and preserve accession IDs and metadata.
2. Define inclusion criteria; avoid mixing unrelated sequence types or incomparable lengths.
3. Use BLAST or another scalable search method to identify candidate pairs before detailed alignments on large datasets.
4. For evolutionary inference, create a multiple sequence alignment and use a suitable phylogenetic method with model selection and branch support.
5. Investigate anomaly flags using taxonomy, metadata, quality information, and independent analyses.
6. Record data accessions, retrieval date, software versions, parameters, and limitations.

**Do not claim:** contamination, genetic engineering, malicious intent, transmission, ancestry, or biological function from these outputs alone. This notebook is an exploratory prototype, not a validated forensic diagnostic system.